# Build Micro Textbook

**Generated by** `splc --lang python/linalg`
**Domain library:** `linalg_graph.py`
**DODA note:** The source `.spl` file is the logical view; this notebook is the physical artifact for the `python/linalg` domain target.

## Inputs

| Parameter | Type | Description |
|-----------|------|-------------|
| `@target` | `TEXT` | — |
| `@style` | `TEXT` | — |
| `@primitive_budget` | `INT` | — |
| `@payoff_weight` | `FLOAT` | — |

In [ ]:
# ── python/linalg target setup — generated by splc ───────────────────────────
import sys, os, json, time
from pathlib import Path

# Locate linalg_graph.py (cookbook/71_linalg_micro_textbook or current dir)
_SEARCH = [
    Path(__file__).parent if "__file__" in dir() else Path("."),
    Path(os.environ.get("LINALG_GRAPH_DIR", ".")),
    Path("."),
]
for _p in _SEARCH:
    _p = _p.resolve()
    if (_p / "linalg_graph.py").exists():
        sys.path.insert(0, str(_p))
        break
import linalg_graph as lg
from linalg_graph import (
    build, acyclic, reducible, minimal, ancestors, restrict,
    productivity_order, in_graph, applications_of, new_primitives,
    first_radical_primitives, both_radical_primitives,
    concept_names, primitive_names,
    gap, learning_path,
)
from style_profiles import style_instruction, get_style_profile, available_styles

# ── LLM helper (configure SPL_MODEL / SPL_LLM_TIMEOUT or replace with your adapter) ──
def _llm_call(prompt: str, model: str | None = None) -> str:
    import subprocess
    _m = model or os.environ.get("SPL_MODEL", "llama3.2")
    # Local models (esp. 8B+ on consumer GPUs) can take well over a minute per
    # call, and refine-loop prompts re-send the full prior section as context —
    # default generously and let SPL_LLM_TIMEOUT override per deployment.
    _timeout = int(os.environ.get("SPL_LLM_TIMEOUT", "600"))
    r = subprocess.run(["ollama", "run", _m], input=prompt,
                       capture_output=True, text=True, timeout=_timeout)
    if r.returncode != 0:
        raise RuntimeError(f"LLM call failed: {r.stderr}")
    return r.stdout.strip()

# ── Math verifier helpers ─────────────────────────────────────────────────────
def _verify_math(section: str) -> str:
    """Returns 'pass' or a description of the failure."""
    try:
        import sympy  # noqa: F401 — presence check
        return "pass"   # TODO: parse worked examples from section and recompute
    except Exception as exc:
        return f"fail: {exc}"

def _shape_check(section: str) -> str:
    """Check matrix dimension compatibility. Returns 'pass' or error."""
    return "pass"  # TODO: parse matrix expressions and check shapes

# ── Timing helpers ────────────────────────────────────────────────────────────
# Bare-name wrappers around time.time() — the SPL SOLVE-template parser does
# not yet support dotted-attribute continuations (module.attr(...)), so the
# .spl source calls these instead of `time.time()` directly. Logged per
# section and for the whole run so executions can be profiled and compared
# (e.g. across distributed workers).
def _now() -> float:
    return time.time()

def _elapsed(start: float) -> float:
    return time.time() - start

# ── Workflow parameters (override before running or use papermill) ────────────
target = 'spectral_theorem'  # TEXT
style = 'textbook'  # TEXT
primitive_budget = 2  # INT
payoff_weight = 1.5  # FLOAT
textbook = ""

# Pre-build the linalg concept graph (reused by all SOLVE/ASSERT cells)
graph = lg.build()
primitives = lg.both_radical_primitives()
print(f"linalg_graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
# ── Prompt templates (mirrors CREATE FUNCTION blocks in .spl) ────────────

WRITE_SECTION_PROMPT = """\
You are an expert linear algebra author.

{style_guide}

Write a self-contained section for the concept: {concept}
Surrounding context: {context}

Follow the structure and length specified in the STYLE GUIDE above exactly.
Use LaTeX for all mathematics.
Output only the section text. No preamble, no metadata.
"""

REFINE_SECTION_PROMPT = """\
You are an expert linear algebra author. Revise the section below based on
the feedback provided. Keep all correct content; fix only what is flagged.
Maintain the style described below throughout the revision.

{style_guide}

SECTION:
{section}

FEEDBACK:
{feedback}

Output the revised section only. No preamble. No commentary.
"""

WRITE_PAYOFF_PROMPT = """\
You are an expert author writing the capstone of a linear algebra micro-textbook.

{style_guide}

The textbook culminates in the concept: {target}
Applications this concept unlocks: {applications}

Write a "Payoff" section that:
1. Explains what {target} achieves and why it is the natural endpoint.
2. Shows how it connects to each listed application domain.
3. Ends with an invitation to explore one application in depth.

Follow the length and structure in the STYLE GUIDE above.
Use LaTeX for mathematics. Output only the section. No preamble.
"""

In [ ]:
# SPL: SOLVE @t_workflow_start FLOAT := _now()
t_workflow_start = _now()
print(f'@t_workflow_start =', t_workflow_start)

In [ ]:
# SPL: SOLVE @style_guide TEXT := style_instruction(@style)
style_guide = style_instruction(style)
print(f'@style_guide =', style_guide)

In [ ]:
# SPL: LOGGING f'Style: {@style}' LEVEL INFO
print('[INFO]', f"Style: {style}")

In [ ]:
# SPL: ASSERT acyclic(@graph)
_assert_result = bool(acyclic(graph))
if not _assert_result:
    raise AssertionError(f"ASSERT failed: 'acyclic(graph)'")
else:
    print(f"✓ ASSERT acyclic(graph)")

In [ ]:
# SPL: ASSERT reducible(@graph, @primitives)
_assert_result = bool(reducible(graph, primitives))
if not _assert_result:
    raise AssertionError(f"ASSERT failed: 'reducible(graph, primitives)'")
else:
    print(f"✓ ASSERT reducible(graph, primitives)")

In [ ]:
# SPL: SOLVE @needed SET := ancestors(@graph, @target)
needed = ancestors(graph, target)
print(f'@needed =', needed)

In [ ]:
# SPL: SOLVE @order LIST := productivity_order(restrict(@graph, @needed), weight=@payoff_weight)
order = productivity_order(restrict(graph, needed), weight=payoff_weight)
print(f'@order =', order)

In [ ]:
# SPL: SOLVE @order_len INT := len(@order)
order_len = len(order)
print(f'@order_len =', order_len)

In [ ]:
# SPL: LOGGING f'Teaching {@order_len} concepts toward {@target}' LEVEL INFO
print('[INFO]', f"Teaching {order_len} concepts toward {target}")

In [ ]:
# SPL: @textbook := ''
textbook = ''
print(f'@textbook =', textbook)

In [ ]:
# SPL: @spl__i := 0
spl__i = 0
print(f'@spl__i =', spl__i)

In [ ]:
# SPL: WHILE @spl__i < @order_len DO
_while_iter = 0
while (spl__i < order_len) and _while_iter < 10:
    _while_iter += 1
    # SPL: SOLVE @concept TEXT := @order[@_i]
    concept = order[spl__i]
    print(f'@concept =', concept)
    # SPL: SOLVE @t_section_start FLOAT := _now()
    t_section_start = _now()
    print(f'@t_section_start =', t_section_start)
    # SPL: LOGGING f'Section {@_i}: {@concept}' LEVEL INFO
    print('[INFO]', f"Section {spl__i}: {concept}")
    # SPL: GENERATE write_section(@concept, @concept, @style_guide) INTO @section
    _prompt = WRITE_SECTION_PROMPT.format(concept=concept, context=concept, style_guide=style_guide)
    section = _llm_call(_prompt)
    print(f'@section:', section[:200] if isinstance(section, str) else section)
    # SPL: ASSERT new_primitives(@section)<=@primitive_budget
    _assert_result = bool(new_primitives(section)<=primitive_budget)
    if not _assert_result:
        print(f"ASSERT failed: 'new_primitives(section)<=primitive_budget' → executing OTHERWISE")
        # SPL: GENERATE refine_section(@section, 'introduce at most one new primitive per section', @style_guide) INTO @section
        _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback='introduce at most one new primitive per section', style_guide=style_guide)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
    else:
        print(f"✓ ASSERT new_primitives(section)<=primitive_budget")
    # SPL: CALL verify_math(@section) INTO @check
    check = _verify_math(section)
    print(f'@check:', check)
    # SPL: EVALUATE @check
    if "fail" in str(check).lower():
        # SPL: GENERATE refine_section(@section, @check, @style_guide) INTO @section
        _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=check, style_guide=style_guide)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
    else:
        # SPL: LOGGING 'Math verified' LEVEL INFO
        print('[INFO]', 'Math verified')
    # SPL: CALL shape_check(@section) INTO @shapes
    shapes = _shape_check(section)
    print(f'@shapes:', shapes)
    # SPL: EVALUATE @shapes
    if "fail" in str(shapes).lower():
        # SPL: GENERATE refine_section(@section, @shapes, @style_guide) INTO @section
        _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=shapes, style_guide=style_guide)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
    else:
        # SPL: LOGGING 'Shapes OK' LEVEL INFO
        print('[INFO]', 'Shapes OK')
    # SPL: SOLVE @section_elapsed FLOAT := _elapsed(@t_section_start)
    section_elapsed = _elapsed(t_section_start)
    print(f'@section_elapsed =', section_elapsed)
    # SPL: LOGGING f'[timing] {@concept}: {@section_elapsed:.1f}s' LEVEL INFO
    print('[INFO]', f"[timing] {concept}: {section_elapsed:.1f}s")
    # SPL: @textbook := @textbook + '\n\n---\n\n' + @section
    textbook = textbook + '\n\n---\n\n' + section
    print(f'@textbook =', textbook)
    # SPL: @spl__i := @spl__i + 1
    spl__i = spl__i + 1
    print(f'@spl__i =', spl__i)
# END WHILE  (ran {_while_iter} iteration(s))

In [ ]:
# SPL: SOLVE @apps LIST := applications_of(@graph, @target)
apps = applications_of(graph, target)
print(f'@apps =', apps)

In [ ]:
# SPL: SOLVE @t_payoff_start FLOAT := _now()
t_payoff_start = _now()
print(f'@t_payoff_start =', t_payoff_start)

In [ ]:
# SPL: GENERATE write_payoff(@target, @apps, @style_guide) INTO @capstone
_prompt = WRITE_PAYOFF_PROMPT.format(target=target, applications=apps, style_guide=style_guide)
capstone = _llm_call(_prompt)
print(f'@capstone:', capstone[:200] if isinstance(capstone, str) else capstone)

In [ ]:
# SPL: SOLVE @payoff_elapsed FLOAT := _elapsed(@t_payoff_start)
payoff_elapsed = _elapsed(t_payoff_start)
print(f'@payoff_elapsed =', payoff_elapsed)

In [ ]:
# SPL: LOGGING f'[timing] payoff: {@payoff_elapsed:.1f}s' LEVEL INFO
print('[INFO]', f"[timing] payoff: {payoff_elapsed:.1f}s")

In [ ]:
# SPL: @textbook := @textbook + '\n\n---\n\n## Payoff\n\n' + @capstone
textbook = textbook + '\n\n---\n\n## Payoff\n\n' + capstone
print(f'@textbook =', textbook)

In [ ]:
# SPL: SOLVE @total_elapsed FLOAT := _elapsed(@t_workflow_start)
total_elapsed = _elapsed(t_workflow_start)
print(f'@total_elapsed =', total_elapsed)

In [ ]:
# SPL: LOGGING f'[timing] workflow total: {@total_elapsed:.1f}s for {@order_len} sections + payoff' LEVEL INFO
print('[INFO]', f"[timing] workflow total: {total_elapsed:.1f}s for {order_len} sections + payoff")

In [ ]:
# SPL: LOGGING f'Textbook complete — style: {@style}' LEVEL INFO
print('[INFO]', f"Textbook complete — style: {style}")

In [ ]:
# SPL: COMMIT @textbook
import json
_output = textbook
_result_path = Path("build_micro_textbook_output.json")
_result_path.write_text(json.dumps(_output, indent=2, default=str))
print(f"Committed to {_result_path}")